# t0d0

* add one more column with the argmax of the q05
* Come up with a more lenient metric that will give you about 70% annotation

# Packages

In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import os
import spatialdata as spd

from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Data

# Locations

In [ ]:
# setting up addresses

# proj_folder       = Path("/coh_labs/yunroseli/Jona/CAR-T/results/cell2location/FirstPass/") # Cohorts
# proj_folder       = Path("/home/janzules/spatial/CAR-T/data/cell2location") # Gemini - first pass files
# proj_folder     = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/cell2location/")
proj_folder   = Path("/coh_labs/yunroseli/Jona/CAR-T/") # Most up to date files

out_figs          = proj_folder / "figures/manual_qc"
adata_out_folder  = proj_folder / "annotated_adata"
adata_out_folder.mkdir(exist_ok=True)
out_figs.mkdir(exist_ok=True)
# Files

zarr_file = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/Caris_Mouse_OME_output/Zarr/concatenated_tissues_FINAL"
adata_ref  = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/annotating_references/predicted_cells_lvl2_epochs_200_Ncells1_decalpha_100_posterior120_bs8192.h5ad"
out_zarr   = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations"

# zarr_file         = proj_folder / "data/zarr/CellCharterClusters"
# adata_ref        = proj_folder / "results/cell2location/C2L_inputs/predicted_cells_lvl2_epochs_200_Ncells1_decalpha_100.h5ad"
# adata_out         = adata_out_folder / "sp_200_epochs_defaultalpha.h5ad" # moved to the save section under annotation


## Processing data

In [3]:
# Load data
adata_vis = sc.read_h5ad(adata_ref)
zdata = spd.read_zarr(zarr_file)

/tmp/ipykernel_3799050/1391359.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zdata = spd.read_zarr(zarr_file)


In [4]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

def normalize_celltype_cols(df: pd.DataFrame) -> pd.DataFrame:
    # strips everything up to and including "_sf_" so columns become just: T_cell, NK_cell, ...
    new_cols = [re.sub(r"^.*?_sf_", "", c) for c in df.columns]
    out = df.copy()
    out.columns = new_cols
    return out

means_df = normalize_celltype_cols(adata_vis.obsm["means_cell_abundance_w_sf"])
q05_df   = normalize_celltype_cols(adata_vis.obsm["q05_cell_abundance_w_sf"])
q95_df   = normalize_celltype_cols(adata_vis.obsm["q95_cell_abundance_w_sf"])

# force same set + same order
common = sorted(set(means_df.columns) & set(q05_df.columns) & set(q95_df.columns))
means_df = means_df[common]
q05_df   = q05_df[common]
q95_df   = q95_df[common]

assert (means_df.columns == q05_df.columns).all()
assert (means_df.columns == q95_df.columns).all()


# Annotating

In [5]:
def annotate_cell2location_dom(
    means_df: pd.DataFrame,
    q05_df: pd.DataFrame,
    q95_df: pd.DataFrame,
    q_cut: float = 0.05,
    dom_cut: float = 1.3,
    dom_cons_cut: float = 0.35,
    sep_cut: float = -np.inf,
    min_total_mean: float = 0.25,
    unknown_label: str = "Unknown",
):
    assert (means_df.columns == q05_df.columns).all()
    assert (means_df.columns == q95_df.columns).all()

    M = means_df.to_numpy()
    Q05 = q05_df.to_numpy()
    Q95 = q95_df.to_numpy()
    n, k = M.shape

    # winner by mean
    win_idx = M.argmax(axis=1)
    win_mean = M[np.arange(n), win_idx]
    win_q05  = Q05[np.arange(n), win_idx]

    total_mean = M.sum(axis=1)

    # runner-up by mean
    top2_idx = np.argpartition(M, -2, axis=1)[:, -2:]
    ru_idx = np.where(top2_idx[:, 0] == win_idx, top2_idx[:, 1], top2_idx[:, 0])
    ru_mean = M[np.arange(n), ru_idx]
    ru_q95  = Q95[np.arange(n), ru_idx]

    dom = win_mean / (ru_mean + 1e-12)
    dom_cons = win_q05 / (ru_q95 + 1e-12)
    sep = win_q05 - ru_q95

    ok = (
        (total_mean >= min_total_mean) &
        (win_q05 >= q_cut) &
        (dom >= dom_cut) &
        (dom_cons >= dom_cons_cut) &
        (sep >= sep_cut)
    )

    celltypes = means_df.columns.to_numpy()
    labels = np.where(ok, celltypes[win_idx], unknown_label)

    out = pd.DataFrame({
        "label": labels,
        "winner_celltype": celltypes[win_idx],
        "winner_mean": win_mean,
        "winner_q05": win_q05,
        "runnerup_celltype": celltypes[ru_idx],
        "runnerup_mean": ru_mean,
        "runnerup_q95": ru_q95,
        "dom_top1_over_top2": dom,
        "dom_conservative": dom_cons,
        "sep_q05_minus_runnerup_q95": sep,
        "total_abundance_mean": total_mean,
        "is_high_conf": ok,
    }, index=means_df.index)

    return out


In [ ]:
ann_perm = annotate_cell2location_dom(
    means_df, q05_df, q95_df,
    q_cut=0.01,
    dom_cut=1.01,
    dom_cons_cut=0.105,
    sep_cut=-np.inf,
    min_total_mean=0.2,
)

ann_strict = annotate_cell2location_dom(
    means_df, q05_df, q95_df,
    q_cut=0.01,
    dom_cut=1.2,
    dom_cons_cut=0.2,
    sep_cut=-np.inf,
    min_total_mean=0.2,
)

# argmax: always assigns, no quality filter
c2l_argmax = means_df.idxmax(axis=1)

for name, labels in [("argmax", c2l_argmax), ("permissive", ann_perm["label"]), ("strict", ann_strict["label"])]:
    labeled = (labels != "Unknown").sum()
    total = len(labels)
    print(f"\n{'='*40}")
    print(f"{name}: {labeled:,} / {total:,} ({labeled/total*100:.1f}%) annotated")
    print(labels.value_counts().to_string())



argmax: 2,088,557 / 2,088,557 (100.0%) annotated
Erythrocyte          550863
N1_like_Neu          449713
Cancer_cell          368277
N2_like_Neu          353025
Classical_Mono        62485
M2_like_Mac           61832
B                     53903
CD8_T                 49447
Fibroblast            25264
NK                    23998
M1_like_Mac           20805
Endothelial           18083
CD4_T                 10776
NKT                   10265
cDC                    9632
Nonclassical_Mono      6483
Treg                   6185
pDC                    5692
Intermediate_Mac       1829

permissive: 1,243,789 / 2,088,557 (59.6%) annotated
label
Unknown              844768
Cancer_cell          360428
Erythrocyte          216175
N2_like_Neu          196264
N1_like_Neu          140862
M2_like_Mac           60145
Classical_Mono        58261
CD8_T                 45578
B                     40768
Fibroblast            24559
NK                    22607
M1_like_Mac           19287
Endothelial           1

In [ ]:
# 1) Reindex to match zarr order
z_adata = zdata.tables["segmentation_counts"]

# 2) Three annotation labels — runner-up is threshold-agnostic (same for all)
z_adata.obs["c2l_argmax"]      = c2l_argmax.reindex(z_adata.obs_names).values
z_adata.obs["c2l_permissive"]  = ann_perm["label"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_strict"]      = ann_strict["label"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_runnerup"]    = ann_perm["runnerup_celltype"].reindex(z_adata.obs_names).values

# 3) Full abundance matrices → .obsm
z_adata.obsm["c2l_means"] = means_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q05"]   = q05_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q95"]   = q95_df.reindex(z_adata.obs_names)


In [ ]:
import numpy as np
import pandas as pd

assert z_adata.n_obs == adata_vis.n_obs, "n_obs mismatch"
assert set(z_adata.obs_names) == set(adata_vis.obs_names), "obs_names sets differ"
print(f"[OK] Row alignment: {z_adata.n_obs:,} cells")

# --- annotation columns ---
for col, src in [
    ("c2l_argmax",     c2l_argmax),
    ("c2l_permissive", ann_perm["label"]),
    ("c2l_strict",     ann_strict["label"]),
    ("c2l_runnerup",   ann_perm["runnerup_celltype"]),
]:
    assert col in z_adata.obs.columns, f"{col} missing from z_adata.obs"
    left  = z_adata.obs[col].astype(str).values
    right = src.reindex(z_adata.obs_names).astype(str).values
    n_mismatch = (left != right).sum()
    n_nan = pd.isna(z_adata.obs[col]).sum()
    assert n_mismatch == 0, f"{col}: {n_mismatch} mismatched values"
    print(f"[OK] {col}: 0 mismatches, {n_nan} NaN")

# --- obsm matrices ---
for key, src_df in [("c2l_means", means_df), ("c2l_q05", q05_df), ("c2l_q95", q95_df)]:
    assert key in z_adata.obsm, f"{key} missing from z_adata.obsm"
    dst = z_adata.obsm[key]
    assert dst.shape == src_df.shape, f"{key}: shape {dst.shape} vs {src_df.shape}"
    dst_arr = dst.loc[z_adata.obs_names].to_numpy() if isinstance(dst, pd.DataFrame) else np.asarray(dst)
    src_arr = src_df.reindex(z_adata.obs_names).to_numpy()
    max_diff = np.nanmax(np.abs(dst_arr - src_arr))
    assert max_diff < 1e-10, f"{key}: max abs diff = {max_diff}"
    print(f"[OK] {key}: shape={dst.shape}, max|diff|={max_diff:.2e}")

# --- label distribution summary ---
print()
for col in ["c2l_argmax", "c2l_permissive", "c2l_strict"]:
    unknown_frac = (z_adata.obs[col] == "Unknown").mean()
    print(f"{col}: {(1-unknown_frac)*100:.1f}% annotated")


[OK] Row alignment: 2,088,557 cells
[OK] c2l_argmax: 0 mismatches, 0 NaN
[OK] c2l_permissive: 0 mismatches, 0 NaN
[OK] c2l_strict: 0 mismatches, 0 NaN
[OK] c2l_runnerup: 0 mismatches, 0 NaN
[OK] c2l_means: shape=(2088557, 19), max|diff|=0.00e+00
[OK] c2l_q05: shape=(2088557, 19), max|diff|=0.00e+00
[OK] c2l_q95: shape=(2088557, 19), max|diff|=0.00e+00

c2l_argmax: 100.0% annotated
c2l_permissive: 59.6% annotated
c2l_strict: 31.6% annotated


In [ ]:
# Drop any stale c2l columns from a previous run before writing
stale = ["c2l_winner", "c2l_is_high_conf", "c2l_label_perm", "c2l_label_strict", "c2l_runnerup_celltype"]
to_drop = [c for c in stale if c in z_adata.obs.columns]
if to_drop:
    z_adata.obs.drop(columns=to_drop, inplace=True)
    print("Dropped:", to_drop)

print("obs columns with c2l:", [c for c in z_adata.obs.columns if c.startswith("c2l_")])


obs columns with c2l: ['c2l_argmax', 'c2l_permissive', 'c2l_strict', 'c2l_runnerup']


## Saving

In [ ]:
# 1) Push the updated table back into the SpatialData object
zdata.tables["segmentation_counts"] = z_adata

# 2) Write to out_zarr (defined in the locations cell)
zdata.write(out_zarr, overwrite=True)
print("Wrote to:", out_zarr)

# 3) Quick verify — reload and spot-check
import spatialdata as spd
zdata_check = spd.read_zarr(out_zarr)
t = zdata_check.tables["segmentation_counts"]


Wrote to: /coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations


In [36]:

print("Reloaded shape:", t.shape)
print("obs columns with c2l:", [c for c in t.obs.columns if c.startswith("c2l_")])
print("obsm keys:", list(t.obsm.keys()))
print(t.obs["c2l_permissive"].value_counts().head(5))

Reloaded shape: (2088557, 19059)
obs columns with c2l: ['c2l_argmax', 'c2l_permissive', 'c2l_strict', 'c2l_runnerup']
obsm keys: ['c2l_q95', 'c2l_q05', 'c2l_means']
c2l_permissive
Unknown        844768
Cancer_cell    360428
Erythrocyte    216175
N2_like_Neu    196264
N1_like_Neu    140862
Name: count, dtype: int64


: 